# Segment Diagnostics Notebook

출처/source, channel(source / medium), medium, campaign, language 기준으로 유저 퍼널 차이, 보험 선택률 차이, 가격민감도 차이를 확인하는 노트북입니다.

- 분석 대상: `identity_level != reservation_id`
- 퍼널/보험 유의성: 세그먼트 vs 나머지 전체의 two-proportion z-test, Benjamini-Hochberg FDR 보정
- 가격민감도: 전체 예약 가격 하위 25% vs 상위 25%의 결제 완료율 차이
- 주의: `language=(unknown)`은 예약 기반 customer_id 유저가 많이 섞인 attribution 실패 세그먼트입니다.

In [1]:
from __future__ import annotations

import json
import math
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display, Markdown

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 80)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

BASE = Path("/Users/yangjiyun/Desktop/LYK")
PROCESSED = BASE / "data" / "processed"
OUT = BASE / "analysis1" / "segment_diagnostics"
OUT.mkdir(parents=True, exist_ok=True)

STAGE_ORDER = [
    "anonymous_visit",
    "identified_visit",
    "search_started",
    "car_clicked",
    "booking_started",
    "insurance_viewed",
    "insurance_selected",
    "checkout_started",
    "payment_attempted",
    "payment_completed",
]

USER_DIMENSIONS = {
    "source": "utm_source_first",
    "medium": "utm_medium_first",
    "campaign": "utm_campaign_first",
    "language": "language_first",
}

RESERVATION_DIMENSIONS = {
    "source": "utm_source",
    "medium": "utm_medium",
    "campaign": "utm_campaign",
    "language": "language_resolved",
}

MIN_USERS_FOR_TEST = 50
MIN_RESERVATIONS_FOR_PRICE = 30

print("output directory:", OUT)

output directory: /Users/yangjiyun/Desktop/LYK/analysis1/segment_diagnostics


In [2]:
def clean_segment(value: object) -> str:
    if pd.isna(value):
        return "(unknown)"
    text = str(value).strip()
    return text if text else "(unknown)"


def p_from_z(z: float) -> float:
    if not math.isfinite(z):
        return 1.0
    return math.erfc(abs(z) / math.sqrt(2))


def two_prop_test(x1: float, n1: float, x2: float, n2: float) -> tuple[float, float]:
    if n1 <= 0 or n2 <= 0:
        return 0.0, 1.0
    p1 = x1 / n1
    p2 = x2 / n2
    pooled = (x1 + x2) / (n1 + n2)
    se = math.sqrt(max(pooled * (1 - pooled) * (1 / n1 + 1 / n2), 0))
    if se == 0:
        return 0.0, 1.0
    z = (p1 - p2) / se
    return z, p_from_z(z)


def bh_adjust(p_values: pd.Series) -> pd.Series:
    p = p_values.fillna(1.0).astype(float).to_numpy()
    n = len(p)
    order = np.argsort(p)
    ranked = p[order]
    adjusted = np.empty(n, dtype=float)
    running = 1.0
    for i in range(n - 1, -1, -1):
        rank = i + 1
        running = min(running, ranked[i] * n / rank)
        adjusted[order[i]] = running
    return pd.Series(np.minimum(adjusted, 1.0), index=p_values.index)


def add_test_fields(rows: list[dict], metric_name: str) -> pd.DataFrame:
    df = pd.DataFrame(rows)
    if df.empty:
        return df
    df["q_value"] = bh_adjust(df["p_value"])
    df["significant_fdr_05"] = df["q_value"] < 0.05
    df["direction"] = np.where(df["diff_pp"] > 0, "higher", "lower")
    df["metric"] = metric_name
    return df

print("helper functions loaded")

helper functions loaded


In [3]:
def summarize_user_dimension(user: pd.DataFrame, dim_name: str, dim_col: str) -> pd.DataFrame:
    x = user.copy()
    x["segment"] = x[dim_col].map(clean_segment)
    out = x.groupby("segment", dropna=False).agg(
        users=("canonical_user_key", "count"),
        reservation_users=("has_reservation", "sum"),
        paid_users=("has_paid", "sum"),
        insurance_event_users=("has_insurance_selected_event", "sum"),
        insurance_row_users=("has_reservation_insurance_row", "sum"),
        avg_event_count=("event_count", "mean"),
        avg_reservation_count=("reservation_count", "mean"),
    ).reset_index()
    out.insert(0, "dimension", dim_name)
    out["reservation_rate"] = out["reservation_users"] / out["users"]
    out["paid_user_rate"] = out["paid_users"] / out["users"]
    out["paid_after_reservation_rate"] = out["paid_users"] / out["reservation_users"].replace(0, np.nan)
    out["insurance_event_rate_all_users"] = out["insurance_event_users"] / out["users"]
    out["insurance_row_rate_all_users"] = out["insurance_row_users"] / out["users"]
    out["insurance_row_rate_reservation_users"] = out["insurance_row_users"] / out["reservation_users"].replace(0, np.nan)
    return out.sort_values(["dimension", "paid_users", "users"], ascending=[True, False, False])


def funnel_tests(user: pd.DataFrame, dim_name: str, dim_col: str) -> pd.DataFrame:
    total_users = len(user)
    total_by_stage = {
        stage: int((user["funnel_stage_rank"] >= rank).sum())
        for rank, stage in enumerate(STAGE_ORDER, start=1)
    }
    rows: list[dict] = []
    x = user.copy()
    x["segment"] = x[dim_col].map(clean_segment)
    for segment, g in x.groupby("segment", dropna=False):
        n1 = len(g)
        n2 = total_users - n1
        if n1 == 0 or n2 == 0:
            continue
        for rank, stage in enumerate(STAGE_ORDER, start=1):
            reached = int((g["funnel_stage_rank"] >= rank).sum())
            rest_reached = total_by_stage[stage] - reached
            z, p = two_prop_test(reached, n1, rest_reached, n2)
            rate = reached / n1
            rest_rate = rest_reached / n2
            rows.append({
                "dimension": dim_name,
                "segment": segment,
                "stage": stage,
                "stage_rank": rank,
                "segment_users": n1,
                "segment_reached": reached,
                "segment_rate": rate,
                "rest_rate": rest_rate,
                "diff_pp": rate - rest_rate,
                "z_score": z,
                "p_value": p,
                "meets_min_users": n1 >= MIN_USERS_FOR_TEST,
            })
    return add_test_fields(rows, "funnel_stage_reach")

print("summary and funnel functions loaded")

summary and funnel functions loaded


In [4]:
def insurance_tests(user: pd.DataFrame, dim_name: str, dim_col: str) -> pd.DataFrame:
    rows: list[dict] = []
    x = user.copy()
    x["segment"] = x[dim_col].map(clean_segment)
    total_users = len(x)
    total_event = int(x["has_insurance_selected_event"].sum())
    reservers = x[x["has_reservation"]].copy()
    total_reservers = len(reservers)
    total_row = int(reservers["has_reservation_insurance_row"].sum())

    for segment, g in x.groupby("segment", dropna=False):
        n1 = len(g)
        n2 = total_users - n1
        event = int(g["has_insurance_selected_event"].sum())
        rest_event = total_event - event
        z, p = two_prop_test(event, n1, rest_event, n2)
        rows.append({
            "dimension": dim_name,
            "segment": segment,
            "insurance_metric": "event_selected_all_users",
            "denominator": "all_users",
            "segment_n": n1,
            "segment_successes": event,
            "segment_rate": event / n1 if n1 else np.nan,
            "rest_rate": rest_event / n2 if n2 else np.nan,
            "diff_pp": event / n1 - rest_event / n2 if n1 and n2 else np.nan,
            "z_score": z,
            "p_value": p,
            "meets_min_users": n1 >= MIN_USERS_FOR_TEST,
        })

    reservers["segment"] = reservers[dim_col].map(clean_segment)
    for segment, g in reservers.groupby("segment", dropna=False):
        n1 = len(g)
        n2 = total_reservers - n1
        row = int(g["has_reservation_insurance_row"].sum())
        rest_row = total_row - row
        z, p = two_prop_test(row, n1, rest_row, n2)
        rows.append({
            "dimension": dim_name,
            "segment": segment,
            "insurance_metric": "reservation_insurance_row_reservation_users",
            "denominator": "reservation_users",
            "segment_n": n1,
            "segment_successes": row,
            "segment_rate": row / n1 if n1 else np.nan,
            "rest_rate": rest_row / n2 if n2 else np.nan,
            "diff_pp": row / n1 - rest_row / n2 if n1 and n2 else np.nan,
            "z_score": z,
            "p_value": p,
            "meets_min_users": n1 >= MIN_USERS_FOR_TEST,
        })
    return add_test_fields(rows, "insurance_selection")


def price_sensitivity(res: pd.DataFrame, dim_name: str, dim_col: str, q25: float, q75: float) -> pd.DataFrame:
    x = res.copy()
    x["segment"] = x[dim_col].map(clean_segment)
    x["is_paid"] = x["payment_status"].eq("PAID")
    x["price_bucket"] = np.select(
        [x["final_price_base"] <= q25, x["final_price_base"] >= q75],
        ["low_price_q1", "high_price_q4"],
        default="mid_price_q2_q3",
    )
    rows: list[dict] = []
    for segment, g in x.groupby("segment", dropna=False):
        valid_price = g["final_price_base"].notna() & (g["final_price_base"] > 0)
        gv = g[valid_price]
        low = gv[gv["price_bucket"].eq("low_price_q1")]
        high = gv[gv["price_bucket"].eq("high_price_q4")]
        low_paid = int(low["is_paid"].sum())
        high_paid = int(high["is_paid"].sum())
        low_n = len(low)
        high_n = len(high)
        z, p = two_prop_test(high_paid, high_n, low_paid, low_n)
        low_rate = low_paid / low_n if low_n else np.nan
        high_rate = high_paid / high_n if high_n else np.nan
        all_paid = int(g["is_paid"].sum())
        paid_price = g.loc[g["is_paid"] & valid_price, "final_price_base"]
        unpaid_price = g.loc[~g["is_paid"] & valid_price, "final_price_base"]
        rows.append({
            "dimension": dim_name,
            "segment": segment,
            "reservations": len(g),
            "priced_reservations": len(gv),
            "paid_reservations": all_paid,
            "paid_reservation_rate": all_paid / len(g) if len(g) else np.nan,
            "insurance_row_rate_reservations": (g["insurance_row_count"].fillna(0) > 0).mean(),
            "avg_price_base": gv["final_price_base"].mean(),
            "median_price_base": gv["final_price_base"].median(),
            "avg_paid_price_base": paid_price.mean(),
            "avg_unpaid_price_base": unpaid_price.mean(),
            "low_price_reservations": low_n,
            "low_price_paid_rate": low_rate,
            "high_price_reservations": high_n,
            "high_price_paid_rate": high_rate,
            "high_minus_low_paid_rate": high_rate - low_rate if pd.notna(high_rate) and pd.notna(low_rate) else np.nan,
            "price_sensitivity_z": z,
            "price_sensitivity_p_value": p,
            "meets_min_reservations": len(g) >= MIN_RESERVATIONS_FOR_PRICE,
            "meets_min_low_high": low_n >= 10 and high_n >= 10,
        })
    out = pd.DataFrame(rows)
    if out.empty:
        return out
    out["price_sensitivity_q_value"] = bh_adjust(out["price_sensitivity_p_value"])
    out["price_signal"] = np.select(
        [
            out["high_minus_low_paid_rate"].ge(-0.03) & out["meets_min_low_high"],
            out["high_minus_low_paid_rate"].le(-0.08) & out["meets_min_low_high"],
        ],
        ["price_resilient", "price_sensitive"],
        default="inconclusive",
    )
    return out.sort_values(["dimension", "paid_reservations", "reservations"], ascending=[True, False, False])

print("insurance and price functions loaded")

insurance and price functions loaded


## 1. Load and Prepare Data

In [5]:
user_raw = pd.read_csv(PROCESSED / "user_funnel_state.csv", low_memory=False)
res_raw = pd.read_csv(PROCESSED / "reservation_funnel_state.csv", low_memory=False)

user = user_raw[user_raw["identity_level"].ne("reservation_id")].copy()
res = res_raw[res_raw["identity_level"].ne("reservation_id")].copy()

numeric_user_cols = [
    "funnel_stage_rank",
    "event_count",
    "reservation_count",
    "paid_reservation_count",
    "insurance_row_count",
]
for col in numeric_user_cols:
    user[col] = pd.to_numeric(user[col], errors="coerce").fillna(0)

user["has_reservation"] = user["reservation_count"] > 0
user["has_paid"] = user["paid_reservation_count"] > 0
user["has_insurance_selected_event"] = user["has_insurance_selected_event"].fillna(False).astype(bool)
user["has_reservation_insurance_row"] = user["has_reservation_insurance_row"].fillna(False).astype(bool)
user["channel_medium_first"] = user["utm_source_first"].map(clean_segment) + " / " + user["utm_medium_first"].map(clean_segment)

USER_DIMENSIONS_WITH_CHANNEL = dict(USER_DIMENSIONS)
USER_DIMENSIONS_WITH_CHANNEL["channel"] = "channel_medium_first"

language_map = user.set_index("canonical_user_key")["language_first"].map(clean_segment).to_dict()
res["language_resolved"] = res["canonical_user_key"].map(language_map).fillna("(unknown)")
res["final_price_base"] = pd.to_numeric(res["final_price_base"], errors="coerce")
res["insurance_row_count"] = pd.to_numeric(res["insurance_row_count"], errors="coerce").fillna(0)
res["channel"] = res["utm_source"].map(clean_segment) + " / " + res["utm_medium"].map(clean_segment)

RESERVATION_DIMENSIONS_WITH_CHANNEL = dict(RESERVATION_DIMENSIONS)
RESERVATION_DIMENSIONS_WITH_CHANNEL["channel"] = "channel"

load_summary = pd.DataFrame([
    {"metric": "raw_users", "value": len(user_raw)},
    {"metric": "excluded_reservation_id_users", "value": int(user_raw["identity_level"].eq("reservation_id").sum())},
    {"metric": "analysis_users", "value": len(user)},
    {"metric": "raw_reservations", "value": len(res_raw)},
    {"metric": "analysis_reservations", "value": len(res)},
])
display(load_summary)

display(Markdown("### User identity / language quality"))
display(user.assign(language_norm=user["language_first"].map(clean_segment)).groupby(["language_norm", "identity_level"]).size().reset_index(name="users").sort_values("users", ascending=False).head(20))

display(Markdown("### Reservation UTM coverage"))
display(res[["utm_source", "utm_medium", "utm_campaign", "language_resolved"]].apply(lambda s: s.map(clean_segment).value_counts().head(10)))

,metric,value
0,raw_users,68814
1,excluded_reservation_id_users,1176
2,analysis_users,67638
3,raw_reservations,26921
4,analysis_reservations,25745


### User identity / language quality

,language_norm,identity_level,users
23,en,anonymous_session,38050
89,zh,anonymous_session,8710
0,(unknown),customer_id,7014
40,id,anonymous_session,4488
24,en,customer_id,2142
48,ko,anonymous_session,2118
79,th,anonymous_session,1276
90,zh,customer_id,617
31,fr,anonymous_session,371
54,ms,anonymous_session,319


### Reservation UTM coverage

,utm_source,utm_medium,utm_campaign,language_resolved
(unknown),"25,678.0000","25,686.0000","25,686.0000","17,318.0000"
23205597293,NaN,NaN,5.0000,NaN
23688781100,NaN,NaN,8.0000,NaN
23735454208,NaN,NaN,15.0000,NaN
23754383997,NaN,NaN,26.0000,NaN
INFLUENCER,1.0000,1.0000,1.0000,NaN
app,8.0000,NaN,NaN,NaN
cpc,NaN,54.0000,NaN,NaN
de,NaN,NaN,NaN,52.0000
en,NaN,NaN,NaN,"5,842.0000"


## 2. Segment User Summary

세그먼트별 유저 수, 예약 유저 수, 결제 유저 수, 보험 이벤트율을 확인합니다.

In [6]:
user_summary = pd.concat(
    [summarize_user_dimension(user, dim_name, dim_col) for dim_name, dim_col in USER_DIMENSIONS_WITH_CHANNEL.items()],
    ignore_index=True,
)

summary_cols = [
    "dimension", "segment", "users", "reservation_users", "paid_users",
    "reservation_rate", "paid_user_rate", "paid_after_reservation_rate",
    "insurance_event_rate_all_users", "insurance_row_rate_reservation_users",
]
for dimension in USER_DIMENSIONS_WITH_CHANNEL:
    display(Markdown(f"### {dimension}"))
    display(user_summary[(user_summary["dimension"] == dimension) & (user_summary["users"] >= 50)][summary_cols].head(20))

### source

,dimension,segment,users,reservation_users,paid_users,reservation_rate,paid_user_rate,paid_after_reservation_rate,insurance_event_rate_all_users,insurance_row_rate_reservation_users
0,source,(unknown),33504,7775,2210,0.2321,0.0660,0.2842,0.0127,0.9869
1,source,google,29007,1284,387,0.0443,0.0133,0.3014,0.0301,0.9930
2,source,app,1753,570,330,0.3252,0.1882,0.5789,0.1757,0.9947
4,source,{{site_source_name}},185,10,6,0.0541,0.0324,0.6000,0.0270,1.0000
6,source,BFF,69,6,5,0.0870,0.0725,0.8333,0.0580,1.0000
8,source,ig,316,6,2,0.0190,0.0063,0.3333,0.0158,1.0000
9,source,fb,2547,4,1,0.0016,0.0004,0.2500,0.0012,1.0000
11,source,chatgpt.com,52,0,0,0.0000,0.0000,NaN,0.0000,NaN


### medium

,dimension,segment,users,reservation_users,paid_users,reservation_rate,paid_user_rate,paid_after_reservation_rate,insurance_event_rate_all_users,insurance_row_rate_reservation_users
23,medium,(unknown),35307,8309,2514,0.2353,0.0712,0.3026,0.0201,0.9874
24,medium,cpc,16655,1306,402,0.0784,0.0241,0.3078,0.0533,0.9931
26,medium,{{placement}},187,12,8,0.0642,0.0428,0.6667,0.0374,1.0000
27,medium,referral,71,8,7,0.1127,0.0986,0.8750,0.0563,1.0000
32,medium,paid,2699,6,1,0.0022,0.0004,0.1667,0.0015,1.0000
35,medium,demandgen,12379,2,0,0.0002,0.0000,0.0000,0.0002,1.0000
36,medium,Facebook_Right_Column,132,0,0,0.0000,0.0000,NaN,0.0000,NaN


### campaign

,dimension,segment,users,reservation_users,paid_users,reservation_rate,paid_user_rate,paid_after_reservation_rate,insurance_event_rate_all_users,insurance_row_rate_reservation_users
41,campaign,(unknown),34956,8309,2515,0.2377,0.0719,0.3027,0.0203,0.9874
42,campaign,23754383997,2998,446,159,0.1488,0.0530,0.3565,0.1067,1.0000
43,campaign,23735454208,2677,326,122,0.1218,0.0456,0.3742,0.0833,0.9877
44,campaign,23205597293,1470,201,59,0.1367,0.0401,0.2935,0.0932,0.9900
45,campaign,23688781100,2506,261,34,0.1042,0.0136,0.1303,0.0670,0.9962
46,campaign,23683448292,612,70,28,0.1144,0.0458,0.4000,0.0670,0.9714
48,campaign,{{campaign.name}},187,12,8,0.0642,0.0428,0.6667,0.0374,1.0000
49,campaign,BFF_program,71,8,7,0.1127,0.0986,0.8750,0.0563,1.0000
53,campaign,120244167499720340,155,5,1,0.0323,0.0065,0.2000,0.0194,1.0000
57,campaign,seoul_en_2026q2,8684,0,0,0.0000,0.0000,NaN,0.0000,NaN


### language

,dimension,segment,users,reservation_users,paid_users,reservation_rate,paid_user_rate,paid_after_reservation_rate,insurance_event_rate_all_users,insurance_row_rate_reservation_users
78,language,(unknown),7014,7014,1952,1.0000,0.2783,0.2783,0.0000,0.9860
79,language,en,40192,1922,807,0.0478,0.0201,0.4199,0.0293,0.9927
80,language,zh,9327,537,114,0.0576,0.0122,0.2123,0.0369,0.9963
81,language,de,325,26,14,0.0800,0.0431,0.5385,0.0462,1.0000
82,language,ko,2152,25,13,0.0116,0.0060,0.5200,0.0037,1.0000
83,language,fr,400,28,12,0.0700,0.0300,0.4286,0.0375,1.0000
84,language,es,331,28,11,0.0846,0.0332,0.3929,0.0574,1.0000
85,language,nl,145,19,10,0.1310,0.0690,0.5263,0.0759,1.0000
86,language,it,119,18,7,0.1513,0.0588,0.3889,0.1092,0.9444
87,language,ja,219,10,5,0.0457,0.0228,0.5000,0.0274,1.0000


### channel

,dimension,segment,users,reservation_users,paid_users,reservation_rate,paid_user_rate,paid_after_reservation_rate,insurance_event_rate_all_users,insurance_row_rate_reservation_users
141,channel,(unknown) / (unknown),33485,7775,2210,0.2322,0.0660,0.2842,0.0127,0.9869
142,channel,google / cpc,16628,1282,387,0.0771,0.0233,0.3019,0.0524,0.9930
143,channel,app / (unknown),1717,534,304,0.3110,0.1771,0.5693,0.1660,0.9944
146,channel,{{site_source_name}} / {{placement}},185,10,6,0.0541,0.0324,0.6000,0.0270,1.0000
148,channel,BFF / referral,69,6,5,0.0870,0.0725,0.8333,0.0580,1.0000
155,channel,fb / paid,2415,4,1,0.0017,0.0004,0.2500,0.0012,1.0000
158,channel,google / demandgen,12379,2,0,0.0002,0.0000,0.0000,0.0002,1.0000
159,channel,ig / paid,284,2,0,0.0070,0.0000,0.0000,0.0035,1.0000
160,channel,fb / Facebook_Right_Column,132,0,0,0.0000,0.0000,NaN,0.0000,NaN


## 3. Funnel Significance

각 세그먼트의 단계 도달률이 나머지 전체와 유의하게 다른지 봅니다.

In [7]:
funnel = pd.concat(
    [funnel_tests(user, dim_name, dim_col) for dim_name, dim_col in USER_DIMENSIONS_WITH_CHANNEL.items()],
    ignore_index=True,
)

important_stages = ["identified_visit", "car_clicked", "booking_started", "payment_attempted", "payment_completed"]
top_funnel = funnel[
    funnel["meets_min_users"]
    & funnel["significant_fdr_05"]
    & funnel["stage"].isin(important_stages)
].copy()
top_funnel = top_funnel[~((top_funnel["dimension"] == "language") & (top_funnel["segment"] == "(unknown)"))]
top_funnel["abs_diff_pp"] = top_funnel["diff_pp"].abs()

funnel_cols = [
    "dimension", "segment", "stage", "segment_users", "segment_rate",
    "rest_rate", "diff_pp", "q_value", "direction",
]
display(top_funnel.sort_values("abs_diff_pp", ascending=False)[funnel_cols].head(50))

,dimension,segment,stage,segment_users,segment_rate,rest_rate,diff_pp,q_value,direction
41,source,app,identified_visit,1753,0.7461,0.2743,0.4718,0.0000,higher
1461,channel,app / (unknown),identified_visit,1717,0.7408,0.2747,0.4661,0.0000,higher
481,campaign,23205597293,identified_visit,1470,0.6503,0.2785,0.3719,0.0000,higher
551,campaign,23754383997,identified_visit,2998,0.6331,0.2705,0.3626,0.0000,higher
491,campaign,23683448292,identified_visit,612,0.6340,0.2834,0.3506,0.0000,higher
43,source,app,car_clicked,1753,0.5585,0.2188,0.3397,0.0000,higher
1463,channel,app / (unknown),car_clicked,1717,0.5492,0.2192,0.3300,0.0000,higher
531,campaign,23735454208,identified_visit,2677,0.5988,0.2737,0.3251,0.0000,higher
501,campaign,23688781100,identified_visit,2506,0.5966,0.2746,0.3220,0.0000,higher
321,medium,demandgen,identified_visit,12379,0.0305,0.3439,-0.3134,0.0000,lower


In [8]:
for stage in ["booking_started", "payment_attempted", "payment_completed"]:
    x = funnel[
        funnel["meets_min_users"]
        & funnel["significant_fdr_05"]
        & funnel["stage"].eq(stage)
    ].copy()
    x = x[~((x["dimension"] == "language") & (x["segment"] == "(unknown)"))]
    x["abs_diff_pp"] = x["diff_pp"].abs()
    display(Markdown(f"### {stage}"))
    display(x.sort_values("abs_diff_pp", ascending=False)[funnel_cols].head(20))

### booking_started

,dimension,segment,stage,segment_users,segment_rate,rest_rate,diff_pp,q_value,direction
954,language,en,booking_started,40192,0.0478,0.2828,-0.2350,0.0000,lower
414,campaign,(unknown),booking_started,34956,0.2377,0.0421,0.1956,0.0000,higher
234,medium,(unknown),booking_started,35307,0.2353,0.0426,0.1928,0.0000,higher
44,source,app,booking_started,1753,0.3252,0.1383,0.1868,0.0000,higher
1414,channel,(unknown) / (unknown),booking_started,33485,0.2322,0.0559,0.1763,0.0000,higher
4,source,(unknown),booking_started,33504,0.2321,0.0560,0.1761,0.0000,higher
1594,channel,google / demandgen,booking_started,12379,0.0002,0.1752,-0.1751,0.0000,lower
324,medium,demandgen,booking_started,12379,0.0002,0.1752,-0.1751,0.0000,lower
94,source,google,booking_started,29007,0.0443,0.2175,-0.1732,0.0000,lower
1464,channel,app / (unknown),booking_started,1717,0.3110,0.1388,0.1722,0.0000,higher


### payment_attempted

,dimension,segment,stage,segment_users,segment_rate,rest_rate,diff_pp,q_value,direction
48,source,app,payment_attempted,1753,0.1968,0.0424,0.1544,0.0000,higher
1468,channel,app / (unknown),payment_attempted,1717,0.1852,0.0428,0.1424,0.0000,higher
958,language,en,payment_attempted,40192,0.0223,0.0818,-0.0595,0.0000,lower
418,campaign,(unknown),payment_attempted,34956,0.0749,0.0159,0.0590,0.0000,higher
238,medium,(unknown),payment_attempted,35307,0.0741,0.0161,0.0580,0.0000,higher
1598,channel,google / demandgen,payment_attempted,12379,0.0000,0.0568,-0.0568,0.0000,lower
328,medium,demandgen,payment_attempted,12379,0.0000,0.0568,-0.0568,0.0000,lower
98,source,google,payment_attempted,29007,0.0157,0.0695,-0.0538,0.0000,lower
738,campaign,seoul_en_2026q2,payment_attempted,8684,0.0000,0.0532,-0.0532,0.0000,lower
518,campaign,23725606539,payment_attempted,6218,0.0000,0.0511,-0.0511,0.0000,lower


### payment_completed

,dimension,segment,stage,segment_users,segment_rate,rest_rate,diff_pp,q_value,direction
49,source,app,payment_completed,1753,0.1882,0.0400,0.1483,0.0000,higher
1469,channel,app / (unknown),payment_completed,1717,0.1771,0.0403,0.1367,0.0000,higher
959,language,en,payment_completed,40192,0.0201,0.0786,-0.0585,0.0000,lower
419,campaign,(unknown),payment_completed,34956,0.0719,0.0137,0.0582,0.0000,higher
239,medium,(unknown),payment_completed,35307,0.0712,0.0139,0.0573,0.0000,higher
369,medium,referral,payment_completed,71,0.0986,0.0437,0.0548,0.0481,higher
1599,channel,google / demandgen,payment_completed,12379,0.0000,0.0536,-0.0536,0.0000,lower
329,medium,demandgen,payment_completed,12379,0.0000,0.0536,-0.0536,0.0000,lower
99,source,google,payment_completed,29007,0.0133,0.0667,-0.0533,0.0000,lower
739,campaign,seoul_en_2026q2,payment_completed,8684,0.0000,0.0503,-0.0503,0.0000,lower


## 4. Insurance Selection Significance

보험은 두 기준을 봅니다.

- `event_selected_all_users`: 이벤트상 보험 선택
- `reservation_insurance_row_reservation_users`: 예약 유저 중 실제 보험 row 존재

In [9]:
insurance = pd.concat(
    [insurance_tests(user, dim_name, dim_col) for dim_name, dim_col in USER_DIMENSIONS_WITH_CHANNEL.items()],
    ignore_index=True,
)

top_insurance = insurance[insurance["meets_min_users"] & insurance["significant_fdr_05"]].copy()
top_insurance["abs_diff_pp"] = top_insurance["diff_pp"].abs()
insurance_cols = [
    "dimension", "segment", "insurance_metric", "segment_n", "segment_rate",
    "rest_rate", "diff_pp", "q_value", "direction",
]
display(top_insurance.sort_values("abs_diff_pp", ascending=False)[insurance_cols].head(60))

,dimension,segment,insurance_metric,segment_n,segment_rate,rest_rate,diff_pp,q_value,direction
4,source,app,event_selected_all_users,1753,0.1757,0.0202,0.1555,0.0000,higher
219,channel,app / (unknown),event_selected_all_users,1717,0.1660,0.0206,0.1454,0.0000,higher
79,campaign,23754383997,event_selected_all_users,2998,0.1067,0.0204,0.0863,0.0000,higher
152,language,it,event_selected_all_users,119,0.1092,0.0241,0.0851,0.0000,higher
72,campaign,23205597293,event_selected_all_users,1470,0.0932,0.0227,0.0705,0.0000,higher
77,campaign,23735454208,event_selected_all_users,2677,0.0833,0.0218,0.0615,0.0000,higher
165,language,nl,event_selected_all_users,145,0.0759,0.0242,0.0517,0.0005,higher
74,campaign,23688781100,event_selected_all_users,2506,0.0670,0.0226,0.0444,0.0000,higher
73,campaign,23683448292,event_selected_all_users,612,0.0670,0.0239,0.0431,0.0000,higher
41,medium,cpc,event_selected_all_users,16655,0.0533,0.0148,0.0385,0.0000,higher


## 5. Price Sensitivity

전체 예약 가격의 하위 25%와 상위 25% 결제 완료율을 비교합니다. `high_minus_low_paid_rate`가 크게 음수면 고가 구간에서 결제율이 낮은 가격민감 신호입니다.

In [10]:
valid_price = res["final_price_base"].notna() & (res["final_price_base"] > 0)
q25 = float(res.loc[valid_price, "final_price_base"].quantile(0.25))
q75 = float(res.loc[valid_price, "final_price_base"].quantile(0.75))

price = pd.concat(
    [price_sensitivity(res, dim_name, dim_col, q25, q75) for dim_name, dim_col in RESERVATION_DIMENSIONS_WITH_CHANNEL.items()],
    ignore_index=True,
)

price_thresholds = pd.DataFrame([
    {"metric": "low_price_q1_max", "final_price_base": q25},
    {"metric": "high_price_q4_min", "final_price_base": q75},
])
display(price_thresholds)

price_cols = [
    "dimension", "segment", "reservations", "paid_reservation_rate", "avg_price_base",
    "low_price_reservations", "low_price_paid_rate", "high_price_reservations",
    "high_price_paid_rate", "high_minus_low_paid_rate", "price_sensitivity_q_value", "price_signal",
]

eligible_price = price[price["meets_min_reservations"] & price["meets_min_low_high"]].copy()
display(Markdown("### Price resilient candidates"))
display(eligible_price[eligible_price["price_signal"].eq("price_resilient")].sort_values("high_minus_low_paid_rate", ascending=False)[price_cols].head(30))

display(Markdown("### Price sensitive candidates"))
display(eligible_price[eligible_price["price_signal"].eq("price_sensitive")].sort_values("high_minus_low_paid_rate")[price_cols].head(30))

display(Markdown("### Segment coverage for price analysis"))
for dimension in RESERVATION_DIMENSIONS_WITH_CHANNEL:
    display(Markdown(f"#### {dimension}"))
    display(price[price["dimension"].eq(dimension)].sort_values("reservations", ascending=False)[price_cols + ["meets_min_reservations", "meets_min_low_high"]].head(15))

,metric,final_price_base
0,low_price_q1_max,"14,903.5000"
1,high_price_q4_min,"42,345.0000"


### Price resilient candidates

,dimension,segment,reservations,paid_reservation_rate,avg_price_base,low_price_reservations,low_price_paid_rate,high_price_reservations,high_price_paid_rate,high_minus_low_paid_rate,price_sensitivity_q_value,price_signal
17,language,en,5842,0.1518,"39,220.5922",798,0.1454,2088,0.1327,-0.0127,1.0000,price_resilient
19,language,ko,650,0.1031,"14,085.5833",474,0.0844,34,0.0588,-0.0256,1.0000,price_resilient


### Price sensitive candidates

,dimension,segment,reservations,paid_reservation_rate,avg_price_base,low_price_reservations,low_price_paid_rate,high_price_reservations,high_price_paid_rate,high_minus_low_paid_rate,price_sensitivity_q_value,price_signal
18,language,zh,1359,0.0890,"46,819.8182",101,0.1485,652,0.0567,-0.0918,0.0107,price_sensitive


...

,dimension,segment,reservations,paid_reservation_rate,avg_price_base,low_price_reservations,low_price_paid_rate,high_price_reservations,high_price_paid_rate,high_minus_low_paid_rate,price_sensitivity_q_value,price_signal,meets_min_reservations,meets_min_low_high
9,campaign,(unknown),25686,0.1311,"32,354.7374",6426,0.1278,6406,0.0977,-0.0300,0.0000,inconclusive,True,True
10,campaign,23754383997,26,0.1923,"49,527.5385",0,NaN,11,0.0909,NaN,1.0000,inconclusive,False,False
11,campaign,23735454208,15,0.1333,"58,693.6667",1,1.0000,10,0.1000,-0.9000,0.0913,inconclusive,False,False
13,campaign,23688781100,8,0.1250,"62,565.2500",0,NaN,6,0.1667,NaN,1.0000,inconclusive,False,False
14,campaign,23205597293,5,0.2000,"34,145.6000",1,1.0000,2,0.0000,-1.0000,0.1943,inconclusive,False,False
12,campaign,{{campaign.name}},4,0.5000,"6,959.7500",4,0.5000,0,NaN,NaN,1.0000,inconclusive,False,False
15,campaign,INFLUENCER,1,0.0000,"8,049.0000",1,0.0000,0,NaN,NaN,1.0000,inconclusive,False,False


#### language

,dimension,segment,reservations,paid_reservation_rate,avg_price_base,low_price_reservations,low_price_paid_rate,high_price_reservations,high_price_paid_rate,high_minus_low_paid_rate,price_sensitivity_q_value,price_signal,meets_min_reservations,meets_min_low_high
16,language,(unknown),17318,0.1283,"29,805.3229",4891,0.1302,3579,0.0858,-0.0445,0.0000,inconclusive,True,True
17,language,en,5842,0.1518,"39,220.5922",798,0.1454,2088,0.1327,-0.0127,1.0000,price_resilient,True,True
18,language,zh,1359,0.0890,"46,819.8182",101,0.1485,652,0.0567,-0.0918,0.0107,price_sensitive,True,True
19,language,ko,650,0.1031,"14,085.5833",474,0.0844,34,0.0588,-0.0256,1.0000,price_resilient,True,True
24,language,ja,128,0.0703,"19,549.2422",76,0.0526,17,0.0000,-0.0526,1.0000,inconclusive,True,True
21,language,fr,107,0.1121,"35,173.8785",9,0.0000,22,0.0455,0.0455,1.0000,inconclusive,True,False
22,language,es,66,0.1667,"28,036.7121",12,0.0000,6,0.1667,0.1667,1.0000,inconclusive,True,False
23,language,nl,56,0.1786,"31,395.8929",6,0.1667,12,0.1667,0.0000,1.0000,inconclusive,True,False
20,language,de,52,0.2692,"22,799.8846",16,0.3125,3,0.3333,0.0208,1.0000,inconclusive,True,False
25,language,it,47,0.1489,"39,430.7660",9,0.1111,13,0.0000,-0.1111,1.0000,inconclusive,True,False


#### channel

,dimension,segment,reservations,paid_reservation_rate,avg_price_base,low_price_reservations,low_price_paid_rate,high_price_reservations,high_price_paid_rate,high_minus_low_paid_rate,price_sensitivity_q_value,price_signal,meets_min_reservations,meets_min_low_high
46,channel,(unknown) / (unknown),25678,0.1311,"32,356.4501",6424,0.1278,6405,0.0977,-0.0301,0.0000,inconclusive,True,True
47,channel,google / cpc,54,0.1667,"52,580.9444",2,1.0000,29,0.1034,-0.8966,0.0021,inconclusive,True,False
49,channel,app / (unknown),8,0.1250,"26,860.1250",2,0.0000,1,0.0000,0.0000,1.0000,inconclusive,False,False
48,channel,{{site_source_name}} / {{placement}},4,0.5000,"6,959.7500",4,0.5000,0,NaN,NaN,1.0000,inconclusive,False,False
50,channel,INFLUENCER / INFLUENCER,1,0.0000,"8,049.0000",1,0.0000,0,NaN,NaN,1.0000,inconclusive,False,False


## 6. Save Outputs

In [11]:
user_summary.to_csv(OUT / "segment_user_summary.csv", index=False, encoding="utf-8-sig")
funnel.to_csv(OUT / "segment_funnel_significance.csv", index=False, encoding="utf-8-sig")
insurance.to_csv(OUT / "segment_insurance_significance.csv", index=False, encoding="utf-8-sig")
price.to_csv(OUT / "segment_price_sensitivity.csv", index=False, encoding="utf-8-sig")

diagnostics = {
    "analysis_users": int(len(user)),
    "analysis_reservations": int(len(res)),
    "price_q25_final_price_base": q25,
    "price_q75_final_price_base": q75,
    "min_users_for_test": MIN_USERS_FOR_TEST,
    "min_reservations_for_price": MIN_RESERVATIONS_FOR_PRICE,
    "dimensions": list(USER_DIMENSIONS_WITH_CHANNEL.keys()),
    "notes": [
        "language=(unknown) is structurally biased because many customer_id-only reservation users have event_count=0.",
        "price sensitivity compares paid rate in global low-price Q1 reservations vs high-price Q4 reservations.",
    ],
}
(OUT / "segment_diagnostics_summary.json").write_text(json.dumps(diagnostics, ensure_ascii=False, indent=2), encoding="utf-8")

display(pd.DataFrame([diagnostics]))
print("Saved CSV/JSON outputs to", OUT)

,analysis_users,analysis_reservations,price_q25_final_price_base,price_q75_final_price_base,min_users_for_test,min_reservations_for_price,dimensions,notes
0,67638,25745,"14,903.5000","42,345.0000",50,30,"[source, medium, campaign, language, channel]",[language=(unknown) is structurally biased bec...


Saved CSV/JSON outputs to /Users/yangjiyun/Desktop/LYK/analysis1/segment_diagnostics


## Interpretation Notes

- `language=(unknown)`은 성과가 좋은 언어가 아니라, 이벤트가 붙지 않은 예약 기반 customer 세그먼트로 보는 것이 안전합니다.
- 예약 row의 UTM은 대부분 `(unknown)`이라 source/medium/campaign 기준 가격민감도는 현재 데이터만으로 해석력이 낮습니다.
- 보험 선택률은 실제 보험 row보다 `has_insurance_selected_event`가 세그먼트 차이를 더 잘 보여줍니다.